# Market Basket Analysis using Apriori Algorithm
This notebook performs market basket analysis on a retail transaction dataset to identify frequently purchased product combinations.

## 1. Load and Preprocess the Dataset

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt
import seaborn as sns

# Each non-empty row is one basket. Use the demo or a headerless grocery basket CSV.
import csv
from pathlib import Path

input_file = Path("demo-baskets.csv")
with input_file.open(encoding="utf-8-sig", newline="") as stream:
    transactions = [
        [item.strip() for item in row if item.strip()]
        for row in csv.reader(stream) if any(item.strip() for item in row)
    ]

# Encode the transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Dataset loaded with {len(transactions)} transactions.")
df.head()

## 2. Apply the Apriori Algorithm
Extract frequent itemsets with a minimum support (e.g., 0.003, or 0.3% of baskets).

In [ ]:
# Apply Apriori
frequent_itemsets = apriori(df, min_support=0.003, use_colnames=True)
print(f"Found {len(frequent_itemsets)} frequent itemsets.")
frequent_itemsets.head()

## 3. Generate Association Rules
Generate rules using support, confidence, and lift.

In [ ]:
# Generate rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
print(f"Generated {len(rules)} association rules.")
rules.head()

## 4. Sort Rules based on Lift
Identify strong associations by sorting the rules.

In [ ]:
# Sort rules by lift
strong_rules = rules.sort_values('lift', ascending=False)
strong_rules.head(10)

## 5. Export Results to CSV

In [ ]:
# Export frequent itemsets and rules
frequent_itemsets.to_csv('frequent_itemsets.csv', index=False)
strong_rules.to_csv('association_rules.csv', index=False)
print("Files exported: frequent_itemsets.csv, association_rules.csv")

## 6. Visualize Strongest Associations

In [ ]:
# Visualize Top 10 rules by lift
top_10_rules = strong_rules.head(10).copy()
top_10_rules['rule'] = top_10_rules['antecedents'].apply(lambda x: ', '.join(list(x))) + ' -> ' + top_10_rules['consequents'].apply(lambda x: ', '.join(list(x)))

plt.figure(figsize=(12, 6))
sns.barplot(x='lift', y='rule', data=top_10_rules, palette='viridis')
plt.title('Top 10 Association Rules by Lift')
plt.xlabel('Lift')
plt.ylabel('Association Rule')
plt.tight_layout()
plt.savefig('top_rules_lift.png')
plt.show()

# Scatter plot of Support vs Confidence
plt.figure(figsize=(10, 6))
sns.scatterplot(x='support', y='confidence', size='lift', hue='lift', data=rules, palette='plasma')
plt.title('Support vs Confidence (Size/Color by Lift)')
plt.grid(True)
plt.savefig('support_vs_confidence.png')
plt.show()